# Note 05 — Event Validation & Comparative Interpretation

## 1. Research Question

Các candidate events được phát hiện ở Note 04 có thực sự phản ánh những thay đổi đáng chú ý trong electricity generation mix hay chỉ là biến động do quy mô hệ thống hoặc denominator?

## 2. Objective

- Kiểm định các candidate events bằng generation thực tế.
- Đánh giá quy mô và hướng thay đổi của electricity source.
- Kiểm tra sự thay đổi của tổng generation trước và sau event.
- Xác định liệu một source tăng có đi kèm với sự suy giảm của source khác hay không.
- Kiểm tra persistence của các thay đổi để phân biệt biến động ngắn hạn và structural shift.
- So sánh các event đã được validation để xác định những trường hợp có ý nghĩa phân tích cao.

## 3. Approach

Kiểm tra candidate events từ Note 04 theo nhiều chiều: generation thực tế → tổng generation → source comparison → replacement pattern → persistence → comparative interpretation.

## 4. Expected Outcome

Xác định được những candidate events có bằng chứng đủ rõ để được xem là các thay đổi đáng chú ý trong electricity mix và sử dụng chúng làm evidence cho câu chuyện Energy Transition.

## 5. Link to Next Note

Các event đã được validation sẽ cung cấp bằng chứng cụ thể cho phân tích **Country Energy Transition Profiles** trong Note 06.

In [ ]:
# ============================================================
# NOTE 5 – EVENT VALIDATION & COMPARATIVE INTERPRETATION
#
# RQ4: Các sự kiện bất thường được phát hiện có đại diện cho
# những kiểu chuyển dịch cơ cấu điện khác nhau hay không?
#
# Câu hỏi con:
# Khi một nguồn điện giảm mạnh, nguồn nào thay thế nó
# và sự thay thế đó có duy trì sau năm xảy ra sự kiện hay không?
#
# Mục đích:
#   - So sánh các event tiêu biểu trên cùng một framework.
#   - Từ đó xác định các kiểu chuyển dịch năng lượng.
# ============================================================

import pandas as pd
from pathlib import Path

relative_path = Path("checkpoints/note04_anomaly_event.pkl")

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / relative_path).is_file()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Không tìm thấy file: {relative_path}"
    )
df_path = project_root / relative_path

note04_checkpoint = (
    project_root / "checkpoints/note04_anomaly_event.pkl"
)

note04_state = pd.read_pickle(note04_checkpoint)

df = note04_state["df"]
generation_sources = note04_state["generation_sources"]
system_sources = note04_state["system_sources"]

event_ranking = note04_state["event_ranking"]
persistence_comparison = note04_state["persistence_comparison"]

japan_event_summary = note04_state["japan_event_summary"]

print("=" * 70)
print("NOTE 5 – EVENT VALIDATION & COMPARATIVE INTERPRETATION")
print("=" * 70)

print()
print("RQ4:")
print("Các sự kiện bất thường được phát hiện có đại diện cho")
print("những kiểu chuyển dịch cơ cấu điện khác nhau hay không?")

print()
print("Câu hỏi con:")
print("Khi một nguồn điện giảm mạnh, nguồn nào thay thế nó")
print("và sự thay thế đó có duy trì sau năm xảy ra sự kiện hay không?")

print()
print("Dataset shape:", df.shape)

print()
print("Các event được Note 4 lựa chọn:")
print(event_ranking)

print()
print("Note 5 sẽ không phát hiện anomaly mới.")
print("Thay vào đó, Note 5 sẽ VALIDATE và SO SÁNH các event")
print("trên cùng một framework.")

NOTE 5 – EVENT VALIDATION & COMPARATIVE INTERPRETATION

RQ4:
Các sự kiện bất thường được phát hiện có đại diện cho
những kiểu chuyển dịch cơ cấu điện khác nhau hay không?

Câu hỏi con:
Khi một nguồn điện giảm mạnh, nguồn nào thay thế nó
và sự thay thế đó có duy trì sau năm xảy ra sự kiện hay không?

Dataset shape: (104203, 24)

Các event được Note 4 lựa chọn:
                Country  Event_year  Total_generation_TWh  \
0        United Kingdom        2016                339.18   
1                 Japan        2012               1106.88   
2           Netherlands        2021                120.89   
3               Türkiye        2019                300.18   
4  United Arab Emirates        2023                167.53   

  Largest_source_change  Largest_share_change_pp  \
0                  Coal                  -13.350   
1               Nuclear                  -13.130   
2                   Gas                  -12.568   
3                   Gas                  -11.733   
4          

In [2]:
# ============================================================
# XÁC ĐỊNH NGUỒN GIẢM VÀ NGUỒN THAY THẾ
#
# Với mỗi event, nguồn điện nào giảm mạnh nhất và nguồn nào
# tăng để bù phần thiếu hụt?
#
# Mục đích:
#   - Đưa 5 event về cùng một cách đo.
#   - Xác định nguồn giảm chính.
#   - Xác định các nguồn tăng chính.
#   - Chưa kết luận persistence ở bước này.
# ============================================================

candidate_events = [
    ("United Kingdom", 2016),
    ("Japan", 2012),
    ("Netherlands", 2021),
    ("Türkiye", 2019),
    ("United Arab Emirates", 2023),
]

event_comparison_rows = []

for country, event_year in candidate_events:

    # Lấy dữ liệu năm trước và năm xảy ra event
    before = df[
        (df["Area"] == country) &
        (df["Area type"] == "Country or economy") &
        (df["Year"] == event_year - 1) &
        (df["Electricity source"].isin(generation_sources))
    ][
        [
            "Electricity source",
            "Generation (TWh)",
            "Share of generation (%)"
        ]
    ].set_index("Electricity source")

    during = df[
        (df["Area"] == country) &
        (df["Area type"] == "Country or economy") &
        (df["Year"] == event_year) &
        (df["Electricity source"].isin(generation_sources))
    ][
        [
            "Electricity source",
            "Generation (TWh)",
            "Share of generation (%)"
        ]
    ].set_index("Electricity source")

    # Chỉ so sánh các nguồn xuất hiện ở cả hai năm
    common_sources = before.index.intersection(during.index)

    comparison = pd.DataFrame({
        "Generation_before_TWh": before.loc[
            common_sources, "Generation (TWh)"
        ],
        "Generation_event_TWh": during.loc[
            common_sources, "Generation (TWh)"
        ],
        "Share_before_pct": before.loc[
            common_sources, "Share of generation (%)"
        ],
        "Share_event_pct": during.loc[
            common_sources, "Share of generation (%)"
        ],
    })

    comparison["Delta_generation_TWh"] = (
        comparison["Generation_event_TWh"]
        - comparison["Generation_before_TWh"]
    )

    comparison["Delta_share_pp"] = (
        comparison["Share_event_pct"]
        - comparison["Share_before_pct"]
    )

    # Nguồn giảm mạnh nhất theo TWh
    largest_decrease = (
        comparison["Delta_generation_TWh"]
        .sort_values()
        .iloc[0]
    )

    decrease_source = (
        comparison["Delta_generation_TWh"]
        .sort_values()
        .index[0]
    )

    # Các nguồn tăng, xếp theo TWh giảm dần
    increases = (
        comparison[
            comparison["Delta_generation_TWh"] > 0
        ][["Delta_generation_TWh", "Delta_share_pp"]]
        .sort_values(
            "Delta_generation_TWh",
            ascending=False
        )
    )

    # Lưu toàn bộ thay đổi của event
    for source, row in comparison.sort_values(
        "Delta_generation_TWh"
    ).iterrows():

        event_comparison_rows.append({
            "Country": country,
            "Event_year": event_year,
            "Electricity_source": source,
            "Delta_generation_TWh": row["Delta_generation_TWh"],
            "Delta_share_pp": row["Delta_share_pp"],
            "Role": (
                "Largest decrease"
                if source == decrease_source
                else (
                    "Increase"
                    if row["Delta_generation_TWh"] > 0
                    else "Decrease"
                )
            )
        })

event_comparison = pd.DataFrame(event_comparison_rows)

print("=" * 70)
print("EVENT SOURCE COMPARISON")
print("=" * 70)

print()
print("Nguồn giảm mạnh nhất của từng event:")
largest_decreases = (
    event_comparison[
        event_comparison["Role"] == "Largest decrease"
    ]
    [
        [
            "Country",
            "Event_year",
            "Electricity_source",
            "Delta_generation_TWh",
            "Delta_share_pp"
        ]
    ]
)

print(
    largest_decreases
    .round(3)
    .to_string(index=False)
)

print()
print("Các nguồn tăng trong từng event:")
increases_only = event_comparison[
    event_comparison["Role"] == "Increase"
].sort_values(
    ["Country", "Delta_generation_TWh"],
    ascending=[True, False]
)

print(
    increases_only
    .round(3)
    .to_string(index=False)
)

EVENT SOURCE COMPARISON

Nguồn giảm mạnh nhất của từng event:
             Country  Event_year Electricity_source  Delta_generation_TWh  Delta_share_pp
      United Kingdom        2016               Coal                -45.21         -13.350
               Japan        2012            Nuclear               -144.94         -13.130
         Netherlands        2021                Gas                -15.83         -12.568
             Türkiye        2019                Gas                -35.21         -11.733
United Arab Emirates        2023                Gas                 -9.15         -11.395

Các nguồn tăng trong từng event:
             Country  Event_year Electricity_source  Delta_generation_TWh  Delta_share_pp     Role
               Japan        2012                Gas                 55.66           4.945 Increase
               Japan        2012               Coal                 49.23           4.385 Increase
               Japan        2012       Other fossil                

In [5]:
# ============================================================
# ĐỊNH LƯỢNG CƠ CHẾ THAY THẾ
#
# Phần sản lượng bị mất của nguồn giảm chính được bù bởi
# nguồn nào và mức độ tập trung của sự thay thế ra sao?
#
# Mục đích:
#   1. Đo tổng lượng tăng của các nguồn tăng.
#   2. Tính replacement ratio.
#   3. Xác định nguồn tăng lớn nhất.
#   4. Đo mức đóng góp của nguồn thay thế chính.
# ============================================================

replacement_rows = []

for country, event_year in candidate_events:

    event_data = event_comparison[
        (event_comparison["Country"] == country) &
        (event_comparison["Event_year"] == event_year)
    ].copy()

    # Nguồn giảm mạnh nhất
    decrease = event_data[
        event_data["Role"] == "Largest decrease"
    ].iloc[0]

    loss_twh = abs(decrease["Delta_generation_TWh"])

    # Tất cả nguồn tăng
    increases = event_data[
        event_data["Role"] == "Increase"
    ].copy()

    total_gain_twh = increases[
        "Delta_generation_TWh"
    ].sum()

    # Nguồn tăng lớn nhất
    largest_gain = increases.sort_values(
        "Delta_generation_TWh",
        ascending=False
    ).iloc[0]

    largest_gain_twh = largest_gain[
        "Delta_generation_TWh"
    ]

    # Tỷ lệ bù đắp
    replacement_ratio = (
        total_gain_twh / loss_twh * 100
        if loss_twh > 0
        else None
    )

    # Đóng góp của nguồn tăng lớn nhất
    largest_contribution = (
        largest_gain_twh / total_gain_twh * 100
        if total_gain_twh > 0
        else None
    )

    replacement_rows.append({
        "Country": country,
        "Event_year": event_year,
        "Decreased_source": decrease["Electricity_source"],
        "Loss_TWh": loss_twh,
        "Total_gain_TWh": total_gain_twh,
        "Replacement_ratio_pct": replacement_ratio,
        "Largest_replacement_source": (
            largest_gain["Electricity_source"]
        ),
        "Largest_replacement_TWh": largest_gain_twh,
        "Largest_replacement_contribution_pct": (
            largest_contribution
        )
    })

replacement_analysis = pd.DataFrame(
    replacement_rows
)

print("=" * 70)
print("REPLACEMENT ANALYSIS")
print("=" * 70)

print()
print(
    "Bảng định lượng cơ chế thay thế của 5 event:"
)

print(
    replacement_analysis
    .round(2)
    .to_string(index=False)
)

print()
print("Diễn giải các chỉ số:")
print("- Loss_TWh: sản lượng mất từ nguồn giảm chính.")
print("- Total_gain_TWh: tổng sản lượng tăng của các nguồn khác.")
print("- Replacement_ratio_pct: mức bù đắp tương đối.")
print("- Largest_replacement_source: nguồn tăng lớn nhất.")
print("- Largest_replacement_contribution_pct: tỷ trọng của nguồn tăng lớn nhất trong tổng phần tăng.")

REPLACEMENT ANALYSIS

Bảng định lượng cơ chế thay thế của 5 event:
             Country  Event_year Decreased_source  Loss_TWh  Total_gain_TWh  Replacement_ratio_pct Largest_replacement_source  Largest_replacement_TWh  Largest_replacement_contribution_pct
      United Kingdom        2016             Coal     45.21           49.56                 109.62                        Gas                    43.25                                 87.27
               Japan        2012          Nuclear    144.94          153.01                 105.57                        Gas                    55.66                                 36.38
         Netherlands        2021              Gas     15.83           15.02                  94.88                       Coal                     6.98                                 46.47
             Türkiye        2019              Gas     35.21           35.71                 101.42                      Hydro                    28.89                           

In [6]:
# ============================================================
# KIỂM TRA TÍNH DUY TRÌ SAU EVENT
#
# Sự thay đổi trong năm event là biến động nhất thời
# hay tạo ra thay đổi cơ cấu kéo dài?
#
# Mục đích:
#   - So sánh Pre → Event → Post → Post+2(2 năm sau Event).
#   - Theo dõi nguồn giảm chính.
#   - Theo dõi nguồn thay thế chính.
#   - Không đánh giá bằng một năm duy nhất.
# ============================================================

persistence_rows = []

for _, event in replacement_analysis.iterrows():

    country = event["Country"]
    event_year = int(event["Event_year"])

    decreased_source = event["Decreased_source"]
    replacement_source = event["Largest_replacement_source"]

    for source_role, source in [
        ("Decreased source", decreased_source),
        ("Largest replacement source", replacement_source)
    ]:

        source_data = df[
            (df["Area"] == country) &
            (df["Area type"] == "Country or economy") &
            (df["Electricity source"] == source) &
            (df["Year"].between(event_year - 1, event_year + 2))
        ][
            [
                "Year",
                "Generation (TWh)",
                "Share of generation (%)"
            ]
        ].sort_values("Year")

        for _, row in source_data.iterrows():

            year = int(row["Year"])

            if year == event_year - 1:
                period = "Pre"
            elif year == event_year:
                period = "Event"
            elif year == event_year + 1:
                period = "Post"
            elif year == event_year + 2:
                period = "Post+2"
            else:
                period = "Other"

            persistence_rows.append({
                "Country": country,
                "Event_year": event_year,
                "Source_role": source_role,
                "Electricity_source": source,
                "Period": period,
                "Year": year,
                "Generation_TWh": row["Generation (TWh)"],
                "Share_pct": row["Share of generation (%)"]
            })

persistence_panel = pd.DataFrame(persistence_rows)

print("=" * 70)
print("EVENT PERSISTENCE")
print("=" * 70)

print()
print(
    "Theo dõi nguồn giảm và nguồn thay thế trước – trong – sau event:"
)

print(
    persistence_panel
    .pivot_table(
        index=[
            "Country",
            "Event_year",
            "Source_role",
            "Electricity_source"
        ],
        columns="Period",
        values="Share_pct",
        aggfunc="first"
    )
    .round(3)
    .to_string()
)

print()
print("Lưu ý:")
print("- Pre = năm trước event.")
print("- Event = năm xảy ra event.")
print("- Post = một năm sau event.")
print("- Post+2 = hai năm sau event.")
print()
print("Mục tiêu là xác định thay đổi có được duy trì hay quay trở lại gần mức trước event.")

EVENT PERSISTENCE

Theo dõi nguồn giảm và nguồn thay thế trước – trong – sau event:
Period                                                                          Event    Post  Post+2     Pre
Country              Event_year Source_role                Electricity_source                                
Japan                2012       Decreased source           Nuclear              1.625   1.342   0.000  14.755
                                Largest replacement source Gas                 39.344  38.941  42.427  34.399
Netherlands          2021       Decreased source           Gas                 46.894  39.797  38.421  59.462
                                Largest replacement source Coal                12.077  12.337   7.261   6.248
Türkiye              2019       Decreased source           Gas                 18.282  22.540  32.738  30.015
                                Largest replacement source Hydro               29.529  25.781  16.930  19.907
United Arab Emirates 2023       Decr

In [7]:
# ============================================================
# TỔNG HỢP EVENT VALIDATION
#
# Các anomaly được phát hiện đại diện cho những kiểu
# chuyển dịch cơ cấu điện khác nhau như thế nào?
#
# Mục đích:
#   - Tổng hợp cơ chế thay thế.
#   - Đánh giá persistence.
#   - Phân biệt structural shift và temporary fluctuation.
#   - Tạo bảng kết quả cuối cho Note 5.
# ============================================================

summary_rows = []

for _, event in replacement_analysis.iterrows():

    country = event["Country"]
    event_year = int(event["Event_year"])

    decreased_source = event["Decreased_source"]
    replacement_source = event["Largest_replacement_source"]

    event_data = persistence_panel[
        (persistence_panel["Country"] == country) &
        (persistence_panel["Event_year"] == event_year)
    ]

    decreased = event_data[
        event_data["Source_role"] == "Decreased source"
    ].set_index("Period")["Share_pct"]

    replacement = event_data[
        event_data["Source_role"] == "Largest replacement source"
    ].set_index("Period")["Share_pct"]

    # Mức thay đổi của nguồn giảm sau 2 năm
    pre_decreased = decreased.get("Pre")
    post2_decreased = decreased.get("Post+2")

    # Mức thay đổi của nguồn thay thế sau 2 năm
    pre_replacement = replacement.get("Pre")
    post2_replacement = replacement.get("Post+2")

    # Phân loại persistence
    if pd.isna(post2_decreased) or pd.isna(post2_replacement):
        persistence_type = "Insufficient post-event data"

    elif (
        post2_decreased < pre_decreased and
        post2_replacement > pre_replacement
    ):
        persistence_type = "Persistent structural shift"

    else:
        persistence_type = "Limited / temporary shift"

    summary_rows.append({
        "Country": country,
        "Event_year": event_year,
        "Decreased_source": decreased_source,
        "Largest_replacement_source": replacement_source,
        "Loss_TWh": event["Loss_TWh"],
        "Replacement_ratio_pct": event["Replacement_ratio_pct"],
        "Largest_replacement_contribution_pct":
            event["Largest_replacement_contribution_pct"],
        "Decreased_pre_share_pct": pre_decreased,
        "Decreased_post2_share_pct": post2_decreased,
        "Replacement_pre_share_pct": pre_replacement,
        "Replacement_post2_share_pct": post2_replacement,
        "Persistence_type": persistence_type
    })

event_validation_summary = pd.DataFrame(summary_rows)

print("=" * 70)
print("EVENT VALIDATION SUMMARY")
print("=" * 70)

print()
print("Bảng tổng hợp 5 event:")
print(
    event_validation_summary
    .round(2)
    .to_string(index=False)
)

print()
print("Phân loại persistence:")
print(
    event_validation_summary[
        [
            "Country",
            "Event_year",
            "Persistence_type"
        ]
    ].to_string(index=False)
)

print()
print("KẾT LUẬN BƯỚC 5:")
print(
    "Không phải mọi anomaly đều tạo ra structural shift."
)
print(
    "Cần phân biệt thay đổi cơ cấu kéo dài với biến động ngắn hạn dựa trên diễn biến sau event."
)

EVENT VALIDATION SUMMARY

Bảng tổng hợp 5 event:
             Country  Event_year Decreased_source Largest_replacement_source  Loss_TWh  Replacement_ratio_pct  Largest_replacement_contribution_pct  Decreased_pre_share_pct  Decreased_post2_share_pct  Replacement_pre_share_pct  Replacement_post2_share_pct             Persistence_type
      United Kingdom        2016             Coal                        Gas     45.21                 109.62                                 87.27                    22.39                       5.04                      29.47                        39.40  Persistent structural shift
               Japan        2012          Nuclear                        Gas    144.94                 105.57                                 36.38                    14.76                       0.00                      34.40                        42.43  Persistent structural shift
         Netherlands        2021              Gas                       Coal     15.83          

In [9]:
# ============================================================
# CHUẨN HÓA TRANSITION PATTERN & CHECKPOINT
#
# Các event bất thường đại diện cho những kiểu chuyển dịch
# cơ cấu điện khác nhau như thế nào?
#
# Mục đích:
#   - Chuẩn hóa cách diễn giải 5 event.
#   - Không đồng nhất "nguồn tăng lớn nhất" với "nguồn thay thế duy nhất".
#   - Hoàn thiện kết quả Note 5.
# ============================================================

event_validation_final = event_validation_summary.copy()

def classify_transition(row):
    country = row["Country"]

    if country == "United Kingdom":
        return "Coal → Gas-dominated replacement"

    elif country == "Japan":
        return "Nuclear → Multi-fossil replacement"

    elif country == "Netherlands":
        return "Gas → Multi-source replacement"

    elif country == "Türkiye":
        return "Gas → Hydro-dominated replacement"

    elif country == "United Arab Emirates":
        return "Gas → Clean-source replacement"

    return "Other"

event_validation_final["Transition_pattern"] = (
    event_validation_final.apply(
        classify_transition,
        axis=1
    )
)

# ------------------------------------------------------------
# Tạo bảng kết quả cuối, chỉ giữ các biến phục vụ báo cáo
# ------------------------------------------------------------

event_validation_final = event_validation_final[
    [
        "Country",
        "Event_year",
        "Decreased_source",
        "Largest_replacement_source",
        "Loss_TWh",
        "Replacement_ratio_pct",
        "Largest_replacement_contribution_pct",
        "Persistence_type",
        "Transition_pattern"
    ]
]

print("=" * 70)
print("FINAL EVENT VALIDATION")
print("=" * 70)

print()
print("Kết quả cuối của 5 event:")
print(
    event_validation_final
    .round(2)
    .to_string(index=False)
)

print()
print("Các kiểu chuyển dịch được xác định:")

for _, row in event_validation_final.iterrows():
    print(
        f"- {row['Country']} ({int(row['Event_year'])}): "
        f"{row['Transition_pattern']}"
    )

print()
print("Ý nghĩa:")
print("- UK: thay thế tập trung vào Gas.")
print("- Japan: phần thiếu hụt Nuclear được bù bởi nhiều nguồn fossil.")
print("- Netherlands: phần thay thế phân tán trên nhiều nguồn.")
print("- Türkiye: Hydro đóng vai trò thay thế chính nhưng không duy trì rõ.")
print("- UAE: Nuclear và Solar cùng đóng góp vào thay đổi cơ cấu, nhưng chưa đủ dữ liệu để đánh giá persistence 2 năm.")

# ------------------------------------------------------------
# CHECKPOINT NOTE 5
# ------------------------------------------------------------

note05_checkpoint = (
    project_root / "checkpoints/note05_event_validation.pkl"
)

note05_state = {
    "df": df,
    "generation_sources": generation_sources,
    "system_sources": system_sources,
    "event_ranking": event_ranking,
    "replacement_analysis": replacement_analysis,
    "persistence_panel": persistence_panel,
    "event_validation_summary": event_validation_summary,
    "event_validation_final": event_validation_final,
}

pd.to_pickle(
    note05_state,
    note05_checkpoint
)

print()
print("=" * 70)
print("NOTE 5 – CHECKPOINT CREATED SUCCESSFULLY")
print("=" * 70)
print("Checkpoint:", note05_checkpoint)
print("Objects saved:", len(note05_state))
print()
print("Note 5 đã hoàn tất:")
print("Anomaly → Replacement → Persistence → Transition pattern")

FINAL EVENT VALIDATION

Kết quả cuối của 5 event:
             Country  Event_year Decreased_source Largest_replacement_source  Loss_TWh  Replacement_ratio_pct  Largest_replacement_contribution_pct             Persistence_type                 Transition_pattern
      United Kingdom        2016             Coal                        Gas     45.21                 109.62                                 87.27  Persistent structural shift   Coal → Gas-dominated replacement
               Japan        2012          Nuclear                        Gas    144.94                 105.57                                 36.38  Persistent structural shift Nuclear → Multi-fossil replacement
         Netherlands        2021              Gas                       Coal     15.83                  94.88                                 46.47  Persistent structural shift     Gas → Multi-source replacement
             Türkiye        2019              Gas                      Hydro     35.21                

## Summary 

Các event đã được validation cho thấy electricity transition không diễn ra đồng nhất giữa các quốc gia.

Note 06 sẽ mở rộng phân tích từ các event riêng lẻ sang **country-level transition profiles** trong giai đoạn 2012–2022.